In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [2]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [3]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [4]:
multiply.name

'multiply'

In [5]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [6]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [2]:
from langchain_ollama import ChatOllama

d:\GEN-AI-Course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Chat model
llm = ChatOllama(model="llama3.2:3b")

In [9]:
llm.invoke("hi")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-08-06T04:58:28.3928432Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12730054600, 'load_duration': 9178559600, 'prompt_eval_count': 26, 'prompt_eval_duration': 1738734000, 'eval_count': 8, 'eval_duration': 1768919000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019fd56e-8923-77a0-9697-847bef82ff24-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

In [10]:
llm_with_tools = llm.bind_tools([multiply])

In [11]:
llm_with_tools

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, model='llama3.2:3b', temperature=0.0), kwargs={'tools': [{'type': 'function', 'function': {'name': 'multiply', 'description': 'Given 2 numbers a and b this tool returns their product', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [13]:
new=llm_with_tools.invoke("can you multiply 3 with 1000")

In [15]:
new.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 1000},
 'id': 'a0bd6d20-2238-4e37-9895-60939c2e4078',
 'type': 'tool_call'}

In [17]:
tool_result=multiply.invoke(new.tool_calls[0])

In [18]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='a0bd6d20-2238-4e37-9895-60939c2e4078')

In [19]:
query = HumanMessage('can you multiply 3 with 1000')

In [21]:
msg = [query]

In [22]:
msg

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [23]:
msg.append(tool_result)

In [24]:
msg

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='3000', name='multiply', tool_call_id='a0bd6d20-2238-4e37-9895-60939c2e4078')]

# Project

## Currency Conversion

In [4]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

In [17]:
@tool
def get_conversion_tool(base_currency:str, target_currency:str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
  response=requests.get(url)
  return response.json()

@tool
def convert_curreny(base_currency_value:int,conversion_rate:Annotated[float, InjectedToolArg])->float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate

In [18]:
get_conversion_tool.invoke({"base_currency":"USD","target_currency":"PKR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1785974401,
 'time_last_update_utc': 'Thu, 06 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1786060801,
 'time_next_update_utc': 'Fri, 07 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'PKR',
 'conversion_rate': 277.7544}

In [19]:
convert_curreny.invoke({"base_currency_value":32,"conversion_rate":277.7544})

8888.1408

In [20]:
# tool binding
llm_with_tools=llm.bind_tools([get_conversion_tool,convert_curreny])

In [21]:
messages = [HumanMessage('What is the conversion factor between PKR and USD, and based on that can you convert 10 usd to pkr')]

In [22]:
messages

[HumanMessage(content='What is the conversion factor between PKR and USD, and based on that can you convert 10 usd to pkr', additional_kwargs={}, response_metadata={})]

In [23]:
ai_messaage = llm_with_tools.invoke(messages)

In [24]:
messages.append(ai_messaage)

In [25]:
messages

[HumanMessage(content='What is the conversion factor between PKR and USD, and based on that can you convert 10 usd to pkr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-08-06T06:04:26.0844336Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21696077100, 'load_duration': 5503325900, 'prompt_eval_count': 257, 'prompt_eval_duration': 8645971000, 'eval_count': 48, 'eval_duration': 7532855000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019fd5ab-f5a1-7543-8d9b-6e471a056d92-0', tool_calls=[{'name': 'get_conversion_tool', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': '852ae140-4d0b-47a5-a6e4-1ba49b700238', 'type': 'tool_call'}, {'name': 'convert_curreny', 'args': {'base_currency_value': '10'}, 'id': 'fbaa583d-ab4b-4418-a8cc-aa24e0dbfcb7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [26]:
messages[1].tool_calls

[{'name': 'get_conversion_tool',
  'args': {'base_currency': 'USD', 'target_currency': 'PKR'},
  'id': '852ae140-4d0b-47a5-a6e4-1ba49b700238',
  'type': 'tool_call'},
 {'name': 'convert_curreny',
  'args': {'base_currency_value': '10'},
  'id': 'fbaa583d-ab4b-4418-a8cc-aa24e0dbfcb7',
  'type': 'tool_call'}]

In [34]:
import json

In [38]:
for tool_call in messages[1].tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_tool':
    tool_msg1=get_conversion_tool.invoke(tool_call)
	# fetch this conversion rate
    conversion_rate = json.loads(tool_msg1.content)['conversion_rate']
    #append
    messages.append(tool_msg1)
    # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert_curreny':
    # fetch aargs
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_msg2=convert_curreny.invoke(tool_call)
    messages.append(tool_msg2)

In [39]:
messages

[HumanMessage(content='What is the conversion factor between PKR and USD, and based on that can you convert 10 usd to pkr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-08-06T06:04:26.0844336Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21696077100, 'load_duration': 5503325900, 'prompt_eval_count': 257, 'prompt_eval_duration': 8645971000, 'eval_count': 48, 'eval_duration': 7532855000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019fd5ab-f5a1-7543-8d9b-6e471a056d92-0', tool_calls=[{'name': 'get_conversion_tool', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': '852ae140-4d0b-47a5-a6e4-1ba49b700238', 'type': 'tool_call'}, {'name': 'convert_curreny', 'args': {'base_currency_value': '10', 'conversion_rate': 277.7544}, 'id': 'fbaa583d-ab4b-4418-a8cc-aa24e0dbfcb7', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [40]:
llm_with_tools.invoke(messages).content

'The conversion rate between USD and PKR is approximately 1 USD = 277.7544 PKR.\n\nTo convert 10 USD to PKR, we can multiply 10 by the conversion rate:\n\n10 USD x 277.7544 PKR/USD = 2777.544 PKR'